## import libraries

In [1]:
import geopandas as gpd
import numpy as np
import pandas as pd
import os
from pathlib import Path
import re
from shapely.geometry import MultiPolygon
import warnings

## read datasets

In [2]:
data_root = Path("_data/eurocrops2")

In [3]:
def get_inventory(root_path):
    inventory = []
    
    for country_dir in root_path.iterdir():
        if country_dir.is_dir():
            for file_path in country_dir.glob("*.parquet"):
                parts = file_path.stem.split('_')
                
                country_id = parts[0]   
                year = int(parts[-1])   
                country = country_dir.name  
                
                inventory.append({"country": country, 
                                  "sub_region": country_id, 
                                  "year": year, 
                                  "id": f"{country}_{year}", 
                                  "sub_region_id": f"{country_id}_{year}", 
                                  "path": file_path})
    
    df = pd.DataFrame(inventory)
    
    if not df.empty:
        df = df.sort_values(by=["country", "year"]).reset_index(drop=True)
        
    return df

In [4]:
available_datasets = get_inventory(data_root)
available_datasets

,country,sub_region,year,id,sub_region_id,path
0,at,at,2017,at_2017,at_2017,_data/eurocrops2/at/at_2017.parquet
1,at,at,2018,at_2018,at_2018,_data/eurocrops2/at/at_2018.parquet
2,at,at,2019,at_2019,at_2019,_data/eurocrops2/at/at_2019.parquet
3,be,be2,2016,be_2016,be2_2016,_data/eurocrops2/be/be2_2016.parquet
4,be,be3,2016,be_2016,be3_2016,_data/eurocrops2/be/be3_2016.parquet
5,be,be2,2018,be_2018,be2_2018,_data/eurocrops2/be/be2_2018.parquet
6,be,be3,2018,be_2018,be3_2018,_data/eurocrops2/be/be3_2018.parquet
7,be,be3,2019,be_2019,be3_2019,_data/eurocrops2/be/be3_2019.parquet
8,be,be2,2019,be_2019,be2_2019,_data/eurocrops2/be/be2_2019.parquet
9,bg,bg,2016,bg_2016,bg_2016,_data/eurocrops2/bg/bg_2016.parquet


## select columns of interest

In [6]:
# # see what columns we have on the datasets
# gpd.read_parquet(available_datasets['path'][0]).head(3)

In [7]:
columns = ['original_code', 'geometry', 'cropfield']

In [8]:
def load_and_filter_datasets(dataframe, columns_of_interest):
    processed = {}
    
    for _, row in dataframe.iterrows():
        gdf = gpd.read_parquet(row['path'])
        processed[row['sub_region_id']] = gdf[columns_of_interest].copy()
        
    return processed

In [9]:
datasets = load_and_filter_datasets(available_datasets, columns)
# datasets.keys()

In [10]:
# datasets['fr_2019'].head(2)

## mapping the crops on the dataset

In [11]:
target_cols = ['original_code', 'geometry', 'hcat4_code', 'hcat4_name', 'cropfield']

In [12]:
def load_and_map_datasets(inventory_df, mapping_csv_path, columns_of_interest):
    mapping_df = pd.read_csv(mapping_csv_path)

    mapping_df["original_code"] = mapping_df["original_code"].astype(str).str.strip()
    mapping_df["nuts"] = mapping_df["nuts"].astype(str).str.strip()

    processed = {}

    for _, row in inventory_df.iterrows():
        print(f"Processing {row['sub_region_id']}")
        gdf = gpd.read_parquet(row["path"])

        gdf["original_code"] = gdf["original_code"].astype(str).str.strip()

        country_map = mapping_df[mapping_df["nuts"] == row["sub_region"]]
        if country_map.empty:
            country_map = mapping_df[mapping_df["nuts"] == row["country"]]

        merged_gdf = gdf.merge(country_map[["original_code", 
                                            "hcat4_code", 
                                            "hcat4_name"]], 
                                            on="original_code", 
                                            how="left")
        
        final_cols = [c for c in columns_of_interest if c in merged_gdf.columns]
        processed[row["sub_region_id"]] = merged_gdf[final_cols].copy()
        
    return processed

In [13]:
# datasets['fr_2019']

In [14]:
datasets = load_and_map_datasets(available_datasets, "_data/eurocrops2/eurocrops.csv", target_cols)
# datasets.keys()

Processing fr_2023


In [15]:
# datasets['fr_2018'].head(3)

## eliminate Nan values on geometry

In [16]:
def eliminate_Nan(datasets):
    cleaned_datasets = {}
    
    with warnings.catch_warnings():
        warnings.filterwarnings('ignore', 'GeoSeries.notna', UserWarning)
        
        for key in datasets.keys():
            gdf = datasets[key]
            
            missing_count = gdf.geometry.isna().sum()
            empty_count = gdf.geometry.is_empty.sum()
            
            if missing_count > 0 or empty_count > 0:
                print(f"Cleaning '{key}': Removed {missing_count} NaN and {empty_count} Empty.")
            
            cleaned_gdf = gdf[gdf.geometry.notna() & (~gdf.geometry.is_empty)].copy()
            cleaned_datasets[key] = cleaned_gdf
            
    return cleaned_datasets

In [17]:
datasets = eliminate_Nan(datasets)

Cleaning 'fr_2023': Removed 0 NaN and 2 Empty.


## select crops of interest

maize_corn_popcorn 

winter_barley

potatoes

In [18]:
# # count the most occurent classes
# print(f"Most common classes: {datasets["pt_2018"]['hcat4_name'].value_counts()[:25]} \n")

In [19]:
# define crops of interest
crops = ['winter_barley', 'potatoes', 'maize_corn_popcorn']

def interest_crop(gdf, crop):
    result = gdf[gdf['hcat4_name'] == crop]
    return result, len(result)

crops_dataset = {}

for dataset_id, gdf in datasets.items():

    for crop in crops:

        var_name = f"{crop}_{dataset_id}"

        subset_gdf, subset_size = interest_crop(gdf, crop)

        if subset_size > 0:
            crops_dataset[var_name] = subset_gdf

In [20]:
print(f"Old dataset keys: {datasets.keys()}")
print(f"Number of new datasets created: {len(crops_dataset)}")
print(f"New datasets created: {crops_dataset.keys()}")

Old dataset keys: dict_keys(['fr_2023'])
Number of new datasets created: 3
New datasets created: dict_keys(['winter_barley_fr_2023', 'potatoes_fr_2023', 'maize_corn_popcorn_fr_2023'])


## merging back sub regions and add year and country id

In [21]:
target_countries = available_datasets['country'].unique()
target_countries

array(['fr'], dtype=object)

In [22]:
def merge_sub_regions(dataset_dict, countries=['be']):
    if isinstance(countries, str):
        countries = [countries]
        
    merged_results = {}

    for country in countries:
        pattern = rf'_{country}\d+_' 
        standard_pattern = rf'_{country}_'
        
        sub_region_keys = [k for k in dataset_dict.keys() if re.search(pattern, k)]
        
        if not sub_region_keys:
            standard_keys = [k for k in dataset_dict.keys() if re.search(standard_pattern, k)]
            
            for k in standard_keys:
                parts = k.split('_')
                df_temp = dataset_dict[k].copy()
                df_temp['country_id'] = country
                df_temp['year'] = parts[-1]
                merged_results[k] = df_temp
            continue 

        groups = {}
        for k in sub_region_keys:
            new_key = re.sub(pattern, f'_{country}_', k)
            
            parts = k.split('_')
            year = parts[-1]        
            country_id = country    
            
            df_temp = dataset_dict[k].copy()
            df_temp['country_id'] = country_id
            df_temp['year'] = year
            
            if new_key not in groups:
                groups[new_key] = []
            groups[new_key].append(df_temp)
            
        for target_name, df_list in groups.items():
            merged_results[target_name] = pd.concat(df_list, ignore_index=True)
            
    return merged_results

In [23]:
crops_dataset = merge_sub_regions(crops_dataset, countries=target_countries)
crops_dataset.keys()

dict_keys(['winter_barley_fr_2023', 'potatoes_fr_2023', 'maize_corn_popcorn_fr_2023'])

In [24]:
# crops_dataset['potatoes_fr_2018'].head(3)

## eliminate multipoligons with more than one poligon

In [25]:
def drop_multipoligons(datasets):
    cleaned_results = {}
    
    for name, gdf in datasets.items():
        multi_parts_indices = []
        eliminates = 0
        
        for idx, geom in gdf['geometry'].items():
            if isinstance(geom, MultiPolygon) and len(geom.geoms) > 1:
                multi_parts_indices.append(idx)
                eliminates = eliminates + 1
        
        print(f"Eliminating multipoligon {eliminates} poligons in '{name}'.")
        
        if multi_parts_indices:
            cleaned_gdf = gdf.drop(index=multi_parts_indices)
            cleaned_results[name] = cleaned_gdf.reset_index(drop=True)
        else:
            cleaned_results[name] = gdf.copy()
            
    return cleaned_results

In [26]:
crops_dataset = drop_multipoligons(crops_dataset)

Eliminating multipoligon 63 poligons in 'winter_barley_fr_2023'.
Eliminating multipoligon 29 poligons in 'potatoes_fr_2023'.
Eliminating multipoligon 103 poligons in 'maize_corn_popcorn_fr_2023'.


In [27]:
# crops_dataset['potatoes_fr_2018']

## limit datasets to a certain number of rows

In [28]:
data_size = 500

In [29]:
crops_dataset = {name: df.iloc[:data_size] for name, df in crops_dataset.items()}

## get centroid and long and lat

In [30]:
def access_coords(gpd, index):
    poligon = gpd.geometry.iloc[index]

    coords = list(poligon.exterior.coords)

    coords_numpy = np.array(poligon.exterior.coords)
    return poligon, coords, coords_numpy

In [31]:
def get_centroid(gdf, centroid_as_main=False):
    gdf = gdf.copy()
    local_utm = gdf.estimate_utm_crs()
    gdf['centroid'] = gdf.geometry.to_crs(local_utm).centroid.to_crs(epsg=4326)
    gdf = gdf.to_crs(epsg=4326)
    
    if centroid_as_main: 
        gdf = gdf.set_geometry('centroid')
        
    return gdf

In [32]:
def get_long_lat(gdf):
    gdf['long'] = gdf['centroid'].x
    gdf['lat'] = gdf['centroid'].y
    return gdf

In [33]:
# crops_dataset['potatoes_fr_2018'].head(3)

In [34]:
# # check before applying transformations
# print(crops_dataset['potatoes_fr_2018'].crs)
# crops_dataset['potatoes_fr_2018'].head(3)

In [35]:
for i in crops_dataset:
    print(f"Apply functions on {i}")
    crops_dataset[i] = get_centroid(crops_dataset[i])
    crops_dataset[i] = get_long_lat(crops_dataset[i])

Apply functions on winter_barley_fr_2023
Apply functions on potatoes_fr_2023
Apply functions on maize_corn_popcorn_fr_2023


In [36]:
# # check after applying transformations
# print(crops_dataset['potatoes_fr_2018'].crs)
# crops_dataset['potatoes_fr_2018']

## export data

In [37]:
export_folder = "_data/exports/crop_country_exports"

if not os.path.exists(export_folder):
    os.makedirs(export_folder)
    print(f"Created directory: {export_folder}")

for name in crops_dataset.keys():
    temp_df = pd.DataFrame(crops_dataset[name][['country_id', 'year', 'hcat4_code', 
                                                'hcat4_name', 'long', 'lat']]).copy()
    
    file_path = f"{export_folder}/{name}.csv"
    
    temp_df.to_csv(file_path, index=False)
    print(f"Exported: {file_path}")

Exported: _data/exports/crop_country_exports/winter_barley_fr_2023.csv
Exported: _data/exports/crop_country_exports/potatoes_fr_2023.csv
Exported: _data/exports/crop_country_exports/maize_corn_popcorn_fr_2023.csv
